In [1072]:
import torch
from torchnmf.nmf import NMF
import numpy as np
import pandas as pd

In [1073]:
chr = "chr22"
encoding_450k = pd.read_csv("../tmp/encoded/HM450_{}_4000.csv".format(chr), index_col=0)
encoding_EPIC = pd.read_csv("../tmp/encoded/EPIC_{}_8000.csv".format(chr), index_col=0)

In [1074]:
encoding_450k.shape

(171, 384)

In [1075]:
# test samples: 
# ['AFB182' 'EUB146' 'AFB155' 'EUB046' 'EUB094' 'AFB035' 'EUB134' 'AFB022'
# 'AFB082' 'EUB153' 'EUB120' 'AFB048' 'EUB043' 'EUB003' 'EUB052' 'AFB053'
# 'AFB163']
train_samples = pd.read_csv("../tmp/processed/train_samples.txt", header=None)[0].values
val_samples = pd.read_csv(
    "../tmp/processed/val_samples.txt", header=None)[0].values
test_samples = pd.read_csv(
    "../tmp/processed/test_samples.txt", header=None)[0].values

encoding_450k_test = encoding_450k.loc[test_samples]
encoding_EPIC_test = encoding_EPIC.loc[test_samples]

In [1076]:
encoding_450k_train = encoding_450k.drop(test_samples)
encoding_EPIC_train = encoding_EPIC.drop(test_samples)

In [1077]:
# NMF
# concat encoding_450k_train and encoding_EPIC_train
encoding_train = pd.concat([encoding_450k_train, encoding_EPIC_train], axis=1).to_numpy()
encoding_train = torch.from_numpy(encoding_train).float()

In [1078]:
encoding_train.shape

torch.Size([153, 768])

In [1079]:
rank = 100
model = NMF(encoding_train.shape, rank=rank)
model.fit(encoding_train, tol=1e-7, max_iter=5000, verbose=True)

 21%|██        | 1040/5000 [00:00<00:03, 1192.83it/s, loss=1.02]


1040

In [1080]:
W = model.W.detach().numpy()
H = model.H.detach().numpy()

In [1081]:
H.shape

(153, 100)

In [1082]:
W.shape

(768, 100)

In [1083]:
# compute the reconstruction error
encoding_train_reconstruct = np.matmul(H, W.T)
encoding_train_reconstruct.shape

(153, 768)

In [1084]:
np.abs((encoding_train.numpy() - encoding_train_reconstruct)).mean()

0.001569664

In [1085]:
W1 = W[0:W.shape[0]//2, :]
W2 = W[W.shape[0]//2:, :]

In [1086]:
# 450k -> EPIC
model_450k2EPIC = NMF(encoding_450k_test.shape, W=W1, trainable_W=False, rank=rank)

In [1087]:
model_450k2EPIC.fit(torch.from_numpy(
    encoding_450k_test.to_numpy()), tol=1e-7, max_iter=1000, verbose=True)

 33%|███▎      | 330/1000 [00:00<00:00, 1438.96it/s, loss=0.0513]


330

In [1088]:
# reconstruct EPIC from 450k
encoding_EPIC_reconstruct = np.matmul(model_450k2EPIC.H.detach().numpy(), W2.T)

In [1089]:
# compute the reconstruction error
np.abs((encoding_EPIC_test.to_numpy() - encoding_EPIC_reconstruct)).mean()


0.025610398739693768

In [1090]:
# save the results
df = pd.DataFrame(encoding_EPIC_reconstruct, index=encoding_EPIC_test.index, columns=encoding_EPIC_test.columns)

In [1091]:
df.to_csv("../tmp/encoded/EPIC_{}_8000_reconstruct.csv".format(chr))

In [1092]:
np.sqrt(((encoding_EPIC_test.to_numpy() - encoding_EPIC_reconstruct)**2).mean())

0.034826785815184086